# ── 1. Cài thư viện ───────────────────────────────────────────────────────────
!pip install -q insightface opencv-python-headless google-api-python-client google-auth
# onnxruntime CPU đã có sẵn trên Colab — không cài onnxruntime-gpu (xung đột CUDA)

In [ ]:
# ── 1. Cài thư viện ───────────────────────────────────────────────────────────
!pip install -q insightface onnxruntime opencv-python-headless google-api-python-client google-auth

In [ ]:
# ── 2. Cấu hình ───────────────────────────────────────────────────────────────
FOLDER_IDS       = ["PASTE_FOLDER_ID_HERE"]   # ID thư mục gốc trên Drive
CREDENTIALS_PATH = "/content/credentials.json" # Service Account key
OUTPUT_PATH      = "/content/drive/MyDrive/face_index.json"

In [ ]:
# ── 3. Mount Drive + Upload credentials ───────────────────────────────────────
from google.colab import drive, files
drive.mount('/content/drive')
print("Upload credentials.json:")
files.upload()

In [ ]:
# ── 5. Load InsightFace ───────────────────────────────────────────────────────
import insightface
face_app = insightface.app.FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.5)
print("✅ InsightFace loaded")

In [ ]:
# ── 5b. Test InsightFace với 1 ảnh ───────────────────────────────────────────
from google.colab import files as colab_files
import cv2, numpy as np
from IPython.display import display, Image as IPImage
import io as _io

print("Upload 1 ảnh để test:")
uploaded = colab_files.upload()
fname = list(uploaded.keys())[0]

img = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_COLOR)
faces = face_app.get(img)

print(f"\n✅ Phát hiện {len(faces)} khuôn mặt")
for i, face in enumerate(faces):
    x1,y1,x2,y2 = face.bbox.astype(int)
    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(img, f"#{i+1} ({face.det_score:.2f})", (x1, y1-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    print(f"  Mặt #{i+1}: bbox={x1},{y1},{x2},{y2} | score={face.det_score:.3f} | embedding 512D ✓")

# Hiển thị ảnh có bbox
_, buf = cv2.imencode('.jpg', img)
display(IPImage(data=buf.tobytes()))

In [ ]:
# ── 5. Load InsightFace ───────────────────────────────────────────────────────
import insightface
face_app = insightface.app.FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.5)
print("✅ InsightFace loaded")

In [ ]:
# ── 6. Quét tất cả ảnh (đệ quy thư mục con) ──────────────────────────────────
IMAGE_MIME  = {'image/jpeg', 'image/png', 'image/webp', 'image/heic', 'image/bmp'}
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_all_recursive(root_id, _depth=0):
    images, subfolders, token = [], [], None
    while True:
        r = drive_svc.files().list(
            q=f"'{root_id}' in parents and trashed=false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        for f in r.get('files', []):
            if f['mimeType'] == FOLDER_MIME:
                subfolders.append(f)
            elif f['mimeType'] in IMAGE_MIME:
                f['folder_id'] = root_id
                images.append(f)
        token = r.get('nextPageToken')
        if not token: break

    print('  ' * _depth + f"📁 {root_id}: {len(images)} ảnh, {len(subfolders)} thư mục con")
    for sub in subfolders:
        images += list_all_recursive(sub['id'], _depth + 1)
    return images

all_files = []
for fid in FOLDER_IDS:
    all_files += list_all_recursive(fid)

# Loại trùng
seen = set()
all_files = [f for f in all_files if not (f['id'] in seen or seen.add(f['id']))]
print(f"\n📷 Tổng: {len(all_files)} ảnh")

In [ ]:
# ── 7. Index khuôn mặt → lưu JSON ────────────────────────────────────────────
DRIVE_VIEW = "https://drive.google.com/file/d/{}/view"

def download_image(file_id):
    req = drive_svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    buf = io.BytesIO()
    dl  = MediaIoBaseDownload(buf, req)
    done = False
    while not done: _, done = dl.next_chunk()
    arr = np.frombuffer(buf.getvalue(), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def extract_embedding(img):
    faces = face_app.get(img)
    if not faces: return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    return face.embedding.tolist()

# Resume nếu đã chạy dở
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f: results = json.load(f)
    done_ids = {r['file_id'] for r in results}
    print(f"Resume: bỏ qua {len(done_ids)} ảnh đã xử lý")
else:
    results, done_ids = [], set()

to_process = [f for f in all_files if f['id'] not in done_ids]
print(f"Cần xử lý: {len(to_process)} ảnh")

errors = []
for f in tqdm(to_process):
    try:
        img = download_image(f['id'])
        emb = extract_embedding(img) if img is not None else None
        results.append({
            'file_id':    f['id'],
            'file_name':  f['name'],
            'folder_id':  f['folder_id'],
            'drive_link': DRIVE_VIEW.format(f['id']),
            'embedding':  emb,
        })
    except Exception as e:
        errors.append({'file': f['name'], 'error': str(e)})

    if len(results) % 50 == 0:
        with open(OUTPUT_PATH, 'w') as out: json.dump(results, out)

with open(OUTPUT_PATH, 'w') as out: json.dump(results, out)

indexed = sum(1 for r in results if r['embedding'])
print(f"\n✅ Xong! {indexed}/{len(results)} ảnh có khuôn mặt | Lỗi: {len(errors)}")
print(f"📁 Đã lưu: {OUTPUT_PATH}")

In [ ]:
# ── 8. Lấy File ID của face_index.json để import vào app ─────────────────────
r = drive_svc.files().list(
    q="name='face_index.json' and trashed=false",
    fields="files(id,name)"
).execute()
for f in r.get('files', []):
    print(f"File ID: {f['id']}")
    print(f"→ Dùng ID này trong app: 'Import từ Colab'")